# Children's Phoneme ASR — Full Pipeline
CNN + BiLSTM + CTC Loss

**Sections:** Install → Data → Features → Model → Train → Inference

## 1. Install Dependencies

In [ ]:
!pip install librosa torch torchaudio -q

## 2. Imports & Config

In [ ]:
import os
import json
import random
import numpy as np
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# ── Config ──────────────────────────────────────────────
AUDIO_DIR   = 'data/audio'       # folder with .wav files
LABELS_JSON = 'data/labels.json' # { 'file1.wav': 'h ɛ l oʊ', ... }
SAMPLE_RATE = 16000
N_MFCC      = 40
BATCH_SIZE  = 16
EPOCHS      = 30
LR          = 1e-3
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## 3. Build Vocabulary from Labels

Reads all IPA tokens in your JSON, assigns each an integer index.
`0` is always reserved for the CTC blank token.

In [ ]:
with open(LABELS_JSON, 'r') as f:
    labels_dict = json.load(f)
# labels_dict expected format:
# { 'file1.wav': 'h ɛ l oʊ', 'file2.wav': 'w ɜ r l d', ... }
# i.e. IPA phones are space-separated strings

# Collect all unique phones
all_phones = set()
for phone_str in labels_dict.values():
    for phone in phone_str.strip().split():
        all_phones.add(phone)

# 0 = CTC blank, phones start at 1
vocab       = ['<blank>'] + sorted(all_phones)
phone2idx   = {p: i for i, p in enumerate(vocab)}
idx2phone   = {i: p for p, i in phone2idx.items()}
NUM_CLASSES = len(vocab)

print(f'Vocab size (including blank): {NUM_CLASSES}')
print(f'Sample phones: {vocab[1:10]}')

## 4. Dataset Class

Loads audio → extracts MFCC+delta+delta2 → encodes label to int tensor.

In [ ]:
class PhonemeDataset(Dataset):
    def __init__(self, labels_dict, audio_dir, phone2idx,
                 sr=16000, n_mfcc=40):
        self.items     = list(labels_dict.items())  # [(filename, phone_str), ...]
        self.audio_dir = audio_dir
        self.phone2idx = phone2idx
        self.sr        = sr
        self.n_mfcc    = n_mfcc

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        filename, phone_str = self.items[idx]
        path = os.path.join(self.audio_dir, filename)

        # ── Load audio ──────────────────────────────────
        y, _ = librosa.load(path, sr=self.sr)

        # ── MFCC + delta + delta-delta ───────────────────
        mfcc   = librosa.feature.mfcc(y=y, sr=self.sr, n_mfcc=self.n_mfcc)
        delta  = librosa.feature.delta(mfcc)
        delta2 = librosa.feature.delta(mfcc, order=2)
        feat   = np.vstack([mfcc, delta, delta2])  # (n_mfcc*3, time)
        feat   = feat.T                             # (time, n_mfcc*3)

        # Per-sample normalization (important for children's variable volumes)
        feat = (feat - feat.mean(axis=0)) / (feat.std(axis=0) + 1e-8)

        feat_tensor = torch.FloatTensor(feat)       # (time, 120)

        # ── Encode label ────────────────────────────────
        phones      = phone_str.strip().split()
        label_tensor = torch.LongTensor(
            [self.phone2idx[p] for p in phones if p in self.phone2idx]
        )

        return feat_tensor, label_tensor

print('Dataset class defined.')

## 5. Collate Function + DataLoader

Audio clips have variable lengths — we pad them to the longest in the batch.
We also track the real lengths so CTC loss isn't confused by padding.

In [ ]:
def collate_fn(batch):
    feats, labels = zip(*batch)

    # Lengths before padding
    feat_lengths  = torch.LongTensor([f.shape[0] for f in feats])
    label_lengths = torch.LongTensor([l.shape[0] for l in labels])

    # Pad features to longest in batch: (batch, max_time, 120)
    feats_padded  = pad_sequence(feats,  batch_first=True, padding_value=0.0)
    # Concatenate labels (CTC expects a flat 1D tensor)
    labels_concat = torch.cat(labels)

    return feats_padded, labels_concat, feat_lengths, label_lengths


# ── Train / Val split ──────────────────────────────────
all_items  = list(labels_dict.items())
random.shuffle(all_items)
split      = int(0.9 * len(all_items))
train_dict = dict(all_items[:split])
val_dict   = dict(all_items[split:])

train_ds = PhonemeDataset(train_dict, AUDIO_DIR, phone2idx)
val_ds   = PhonemeDataset(val_dict,   AUDIO_DIR, phone2idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate_fn)

print(f'Train: {len(train_ds)} samples | Val: {len(val_ds)} samples')

## 6. Model — CNN + BiLSTM

In [ ]:
class CNNBiLSTMPhoneme(nn.Module):
    def __init__(self, n_features=120, num_classes=41,
                 lstm_hidden=256, lstm_layers=2):
        super().__init__()

        # ── CNN frontend ─────────────────────────────────
        # Input: (batch, 1, n_features, time)
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(3,3), padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2,1)),   # halve freq axis, keep time

            nn.Conv2d(32, 64, kernel_size=(3,3), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2,1)),   # halve again

            nn.Dropout2d(0.25),
        )
        # After 2x pool on freq: n_features -> n_features//4
        cnn_out_dim = 64 * (n_features // 4)

        # ── BiLSTM ───────────────────────────────────────
        self.bilstm = nn.LSTM(
            input_size=cnn_out_dim,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        self.fc = nn.Linear(lstm_hidden * 2, num_classes)

    def forward(self, x):
        # x: (batch, time, n_features)
        x = x.unsqueeze(1)             # (batch, 1, time, n_features)
        x = x.permute(0, 1, 3, 2)     # (batch, 1, n_features, time)

        x = self.cnn(x)                # (batch, 64, n_features//4, time)

        b, c, f, t = x.shape
        x = x.permute(0, 3, 1, 2)     # (batch, time, 64, n_features//4)
        x = x.reshape(b, t, c * f)    # (batch, time, cnn_out_dim)

        x, _ = self.bilstm(x)         # (batch, time, lstm_hidden*2)
        x = self.fc(x)                 # (batch, time, num_classes)
        return x


N_FEATURES  = N_MFCC * 3   # mfcc + delta + delta2 = 120
model = CNNBiLSTMPhoneme(
    n_features=N_FEATURES,
    num_classes=NUM_CLASSES
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')

## 7. Training Loop

In [ ]:
ctc_loss  = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=3, factor=0.5, verbose=True
)

best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):

    # ── Train ──────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for feats, labels, feat_lens, label_lens in train_loader:
        feats  = feats.to(DEVICE)
        labels = labels.to(DEVICE)

        logits   = model(feats)                          # (batch, time, C)
        log_prob = logits.log_softmax(-1)
        log_prob = log_prob.permute(1, 0, 2)             # (time, batch, C) for CTC

        loss = ctc_loss(log_prob, labels, feat_lens, label_lens)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        train_loss += loss.item()

    # ── Validate ───────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for feats, labels, feat_lens, label_lens in val_loader:
            feats  = feats.to(DEVICE)
            labels = labels.to(DEVICE)

            logits   = model(feats)
            log_prob = logits.log_softmax(-1).permute(1, 0, 2)
            loss     = ctc_loss(log_prob, labels, feat_lens, label_lens)
            val_loss += loss.item()

    train_loss /= len(train_loader)
    val_loss   /= len(val_loader)
    scheduler.step(val_loss)

    print(f'Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'  ✓ Saved best model (val_loss={val_loss:.4f})')

## 8. Inference — Greedy CTC Decode

Greedy decode: take the argmax at each timestep, collapse repeats, remove blanks.

In [ ]:
def greedy_ctc_decode(logits, idx2phone, blank_idx=0):
    """
    logits: (time, num_classes) — single sample, already on CPU
    Returns: list of predicted phone strings
    """
    indices = logits.argmax(-1).tolist()   # argmax at each timestep

    # Collapse consecutive duplicates, then remove blanks
    phones = []
    prev = None
    for idx in indices:
        if idx != prev:
            if idx != blank_idx:
                phones.append(idx2phone[idx])
            prev = idx

    return phones


def predict(audio_path, model, phone2idx, idx2phone,
            sr=16000, n_mfcc=40, device='cpu'):
    model.eval()

    # Extract features (same as training)
    y, _ = librosa.load(audio_path, sr=sr)
    mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    feat   = np.vstack([mfcc, delta, delta2]).T            # (time, 120)
    feat   = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)  # normalize

    feat_tensor = torch.FloatTensor(feat).unsqueeze(0).to(device)  # (1, time, 120)

    with torch.no_grad():
        logits = model(feat_tensor)                 # (1, time, num_classes)
        logits = logits.squeeze(0).cpu()            # (time, num_classes)

    return greedy_ctc_decode(logits, idx2phone)


# ── Example usage ──────────────────────────────────────
# Load best saved model
model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))

test_file = 'data/audio/sample_test.wav'   # replace with a real file
predicted_phones = predict(test_file, model, phone2idx, idx2phone, device=DEVICE)
print('Predicted phones:', ' '.join(predicted_phones))

## 9. CER Evaluation on Val Set

In [ ]:
def compute_cer(predicted, reference):
    """
    predicted, reference: lists of phone strings
    Returns CER = (S + D + I) / N  (edit distance / ref length)
    """
    p, r = predicted, reference
    dp = [[0]*(len(r)+1) for _ in range(len(p)+1)]
    for i in range(len(p)+1): dp[i][0] = i
    for j in range(len(r)+1): dp[0][j] = j
    for i in range(1, len(p)+1):
        for j in range(1, len(r)+1):
            cost = 0 if p[i-1] == r[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
    return dp[len(p)][len(r)] / max(len(r), 1)


model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))
model.eval()

total_cer, n = 0.0, 0
for filename, phone_str in val_dict.items():
    path = os.path.join(AUDIO_DIR, filename)
    if not os.path.exists(path):
        continue
    reference = phone_str.strip().split()
    predicted = predict(path, model, phone2idx, idx2phone, device=DEVICE)
    total_cer += compute_cer(predicted, reference)
    n += 1

print(f'Val CER: {total_cer / n * 100:.2f}%  (over {n} samples)')